# Simulate and explore one robot design

This notebook turns an editable physical design into a robot trajectory, plots
and an animation. You will inspect the initial geometry, run the simulation,
then explore how the body and feet move.

Install `python -m pip install -e ".[analysis,notebooks]"`, open this notebook in
JupyterLab and select that Python environment. Run the cells in order. Controls
update the preview; the simulation runs when you execute its own cell.


## 1. Prepare the simulator

We use float64 arithmetic and the measured CPU configuration for this example.
The base design starts with its feet on the ground and runs for 0.2 seconds.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open from the repository or tutorials directory."
os.chdir(ROOT)
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/walking-robot-mpl")
sys.path.insert(0, str(ROOT / "tutorials"))
import jax
jax.config.update("jax_enable_x64", True)
import yaml
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image
from _widgets import animation_view, preview_design, trajectory_view, show_checks
print("Repository:", ROOT)
print("Python:", sys.executable)
print("Simulation device:", jax.devices())


## 2. Choose the design and inspect its initial pose

Move the angle sliders to see how the legs attach to the body. Adhesive thickness
and damping change the dynamics; this line drawing does not resolve the adhesive
layer. The title shows the applied frequency from the MATLAB design rule.

The controls edit a copy of `matlab_supported.yaml`. The original YAML stays
available as a reference. Parameter meanings and units are listed in
[design parameters](../docs/design_parameters.md).


In [ ]:
from walking_robot.design import RobotDesign
configuration = yaml.safe_load(Path("examples/designs/matlab_supported.yaml").read_text())
thickness = widgets.FloatSlider(value=1, min=1, max=50, step=1, description="Adhesive (µm)", continuous_update=False, style={"description_width": "initial"})
damping = widgets.FloatSlider(value=.01, min=.01, max=.09, step=.02, description="Damping", readout_format=".2f", continuous_update=False)
front = widgets.FloatSlider(value=95, min=75, max=125, step=1, description="Front (°)", continuous_update=False)
hind = widgets.FloatSlider(value=125, min=95, max=150, step=1, description="Hind (°)", continuous_update=False)

def update_design(adhesive, damping_ratio, front_angle, hind_angle):
    configuration["design"].update(adhesive_thickness_um=adhesive, damping_ratio=damping_ratio,
                                   front_angle_deg=front_angle, hind_angle_deg=hind_angle)
    preview_design(RobotDesign(**configuration["design"]))

preview = widgets.interactive_output(update_design, {"adhesive": thickness, "damping_ratio": damping,
                                                    "front_angle": front, "hind_angle": hind})
display(widgets.VBox([widgets.HBox([thickness, damping]), widgets.HBox([front, hind]), preview]))


## 3. Save the YAML and run the design

Execute this cell after choosing the parameters. It saves a fresh run directory,
integrates the dynamics, checks physical limits and generates the figures and
HTML animation. The first run includes compilation.

Changing a slider later updates the design preview. Execute this cell again to
simulate that new design; previously generated runs remain in `outputs/`.


In [ ]:
from uuid import uuid4
configuration["simulation"]["duration_s"] = .2
configuration["visualization"].update(formats=["html"], max_frames=150)
OUTPUT = Path("outputs/tutorial_one_design") / uuid4().hex[:12]
OUTPUT.mkdir(parents=True)
DESIGN = OUTPUT / "input_design.yaml"
DESIGN.write_text(yaml.safe_dump(configuration, sort_keys=False))
print(DESIGN.read_text())
result = subprocess.run([sys.executable, "examples/run_design.py", "--design", str(DESIGN), "--output", str(OUTPUT)])
if result.returncode not in (0, 2):
    raise RuntimeError("Simulation failed; inspect the command output above.")
validation = json.loads((OUTPUT / "validation.json").read_text())
print("Saved run:", OUTPUT)
show_checks(validation)


## 4. Inspect the motion and foot contact

Move **Frame** to compare the robot pose with its COM velocity and foot heights.
**Follow robot** keeps the body in view; clear it to view the travelled distance.
The orange lines mark the selected instant. Foot height below zero means
penetration in the compliant contact model; the dashed line is the provisional
penetration limit.

The display samples the trajectory for responsive plotting. The validation
report above checks every integration sample, including peaks between displayed
frames. A passing screen is evidence under those limits, rather than a hardware
validation.


In [ ]:
display(trajectory_view(OUTPUT / "trajectory.npz", foot_limit_m=validation["limits"]["max_foot_penetration_m"]))


## 5. Play the animation and read the diagnostics

The HTML animation has playback controls. Its frame count and playback settings
produce a slow-motion view; the time label gives simulated time.
The diagnostics summarize the full run, including limits on speeds and heights.


In [ ]:
display(animation_view(OUTPUT / "robot.html"))
display(Image(filename=str(OUTPUT / "diagnostics.png")))


## 6. Compare another design

Return to step 2, change one parameter and rerun steps 3–5. Compare the mean
speed and maximum penetration together: a higher speed is useful only if the
trajectory also passes the physical checks. Mean speed here uses the final
0.1 seconds; longer studies should use a common duration and measurement window.

Each run directory contains its YAML, trajectory, validation, figures and HTML
animation. To compare a parameter grid, continue with
[the sweep notebook](02_compare_designs.ipynb).
